# Conditional SEDD sampling (infilling) + generative perplexity

Conditionally generate text by **pinning** a prefix and suffix (as in `run_sample_cond.py`)
and letting SEDD fill the middle, then score the full sequence under GPT-2 Large
(the same metric as `run_train.py`).

Compares the vanilla analytic sampler against the **`lb_mean_field`** corrector (unified
locally-balanced Metropolis-Hastings), with a matched-NFE comparison. The corrector
auto-detects the SEDD variant from `graph.absorb` — for the default absorb model
`louaaron/sedd-medium` it flips only masked positions to data tokens and uses the
noising-kernel acceptance (1 score call/step). The pinned positions are re-applied via
`proj_fun` on every predictor *and* corrector step, so the prompt is always preserved. See
`.claude/skills/lb_corrector_unified_recipe.md`.

**Requirements:** a CUDA GPU and `flash_attn`. `louaaron/sedd-medium` downloads on first run.

## 1. Setup

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

import sampling
from load_model import load_model

# ---- config ----
MODEL_PATH = 'louaaron/sedd-medium'   # or a local run dir
SEQ_LEN    = 1024                      # SEDD / GPT-2 context length
BATCH_SIZE = 8
SEED       = 42

# Conditioning: text to pin at the start and end; SEDD infills between them.
PREFIX = 'Hi, my name is'
SUFFIX = " and that's why I'm late."

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
model, graph, noise = load_model(MODEL_PATH, device)
model.eval()
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

# External evaluator for generative perplexity (matches run_train.py).
eval_model = GPT2LMHeadModel.from_pretrained('gpt2-large').to(device).eval()
print('loaded SEDD + GPT-2 Large evaluator')

## 2. Build the conditioning projection

`make_projection` mirrors `run_sample_cond.py`: it tokenizes the prefix/suffix, places them at
the first / last positions, and returns a `proj_fun(x)` that re-pins those tokens. Pass that
`proj_fun` to the sampler and it is applied before every update.

In [ ]:
def make_projection(prefix, suffix, batch_size, seq_len=SEQ_LEN):
    prefix_ids = tokenizer(prefix).input_ids
    suffix_ids = tokenizer(suffix).input_ids
    input_ids = prefix_ids + suffix_ids
    input_locs = list(range(len(prefix_ids))) + list(range(seq_len - len(suffix_ids), seq_len))

    input_ids = torch.tensor(input_ids, device=device)[None].repeat(batch_size, 1)

    def proj_fun(x):
        x[:, input_locs] = input_ids
        return x

    return proj_fun, input_locs

proj_fun, input_locs = make_projection(PREFIX, SUFFIX, BATCH_SIZE)
print(f'pinned {len(input_locs)} positions; infilling the remaining {SEQ_LEN - len(input_locs)}')

## 3. Helpers

`generate` wraps `sampling.get_pc_sampler` with the projection; `corrector_steps=0` reproduces
the vanilla conditional sampler exactly. The final `proj_fun` call guarantees the pins survive the
denoising step. `generative_perplexity` is the GPT-2 Large metric from `run_train.py`.

In [ ]:
@torch.no_grad()
def generate(steps, corrector='none', corrector_steps=0, balancing='barker',
             update_fraction=1.0, corrector_t_threshold=0.0, batch_size=BATCH_SIZE,
             seed=SEED):
    if seed is not None:
        torch.manual_seed(seed)
    proj_fun, _ = make_projection(PREFIX, SUFFIX, batch_size)
    sampling_fn = sampling.get_pc_sampler(
        graph, noise, (batch_size, SEQ_LEN), 'analytic', steps, device=device, proj_fun=proj_fun,
        corrector=corrector, corrector_steps=corrector_steps, balancing=balancing,
        update_fraction=update_fraction, corrector_t_threshold=corrector_t_threshold,
    )
    return proj_fun(sampling_fn(model))   # re-pin prompt after denoising


@torch.no_grad()
def generative_perplexity(samples, ppl_batch_size=4):
    """Generative perplexity under GPT-2 Large over the full sequence (lower is better)."""
    batches = max(1, samples.shape[0] // ppl_batch_size)
    total = 0.0
    for i in range(batches):
        s = samples[i * ppl_batch_size:(i + 1) * ppl_batch_size]
        _, logits = eval_model(s, labels=s)[:2]
        logits = logits.transpose(-1, -2)
        ppl = F.cross_entropy(logits[..., :-1], s[..., 1:], reduction='none').mean(dim=-1).exp().mean()
        total += ppl.item()
    return total / batches


# Score calls per corrector step depend on the variant: absorb uses the
# noising-kernel acceptance (1 call), uniform uses Zanella Z_x/Z_y (2 calls).
CORRECTOR_COST = 1 if graph.absorb else 2


def nfe(steps, corrector_steps=0):
    """Score-function calls: analytic predictor = 1/step, corrector = CORRECTOR_COST/step."""
    return steps * (1 + CORRECTOR_COST * corrector_steps)


def show(samples, n=2):
    for txt in tokenizer.batch_decode(samples[:n]):
        print(txt)
        print('=' * 80)

## 4. Vanilla conditional baseline

Every decoded sample should begin with the prefix and end with the suffix.

In [ ]:
STEPS = 128

vanilla = generate(steps=STEPS, corrector='none')
vanilla_ppl = generative_perplexity(vanilla)
print(f'vanilla   | steps={STEPS} NFE={nfe(STEPS)} | gen-ppl = {vanilla_ppl:.3f}')
show(vanilla)

## 5. `lb_mean_field` corrector

The corrector also runs through `proj_fun`, so it refines only the infilled positions and never
overwrites the prompt. On the default absorb model each corrector step is **1 score call** and
flips only masked positions to data tokens (unmasked positions, including the pinned prompt, are
frozen). `K=1` is the recommended default for absorb since unmasks are irreversible.

In [ ]:
K = 1
corrected = generate(steps=STEPS, corrector='lb_mean_field', corrector_steps=K,
                     balancing='barker', update_fraction=1.0)
corrected_ppl = generative_perplexity(corrected)
print(f'lb K={K}    | steps={STEPS} NFE={nfe(STEPS, K)} | gen-ppl = {corrected_ppl:.3f}')
show(corrected)

## 6. Matched-NFE comparison

Hold total NFE fixed: vanilla at `N` levels vs corrector at `~N / (1 + CORRECTOR_COST * K)`
levels with `K=1` (`N/2` for absorb, `N/3` for uniform).

In [ ]:
BUDGET = 384   # target NFE budget

# Matched-NFE level count for K=1, derived from the variant's corrector cost
# (absorb: BUDGET//2, uniform: BUDGET//3).
lb_steps = BUDGET // (1 + CORRECTOR_COST * 1)

configs = [
    dict(label='vanilla',      steps=BUDGET,    corrector='none',          corrector_steps=0),
    dict(label='lb K=1',       steps=lb_steps,  corrector='lb_mean_field', corrector_steps=1),
    dict(label='lb K=1 sqrt',  steps=lb_steps,  corrector='lb_mean_field', corrector_steps=1, balancing='sqrt'),
    dict(label='lb K=1 uf0.5', steps=lb_steps,  corrector='lb_mean_field', corrector_steps=1, update_fraction=0.5),
]

print(f'{"config":<14}{"steps":>7}{"NFE":>7}{"gen-ppl":>10}')
print('-' * 38)
for c in configs:
    label = c.pop('label')
    s = generate(**c)
    ppl = generative_perplexity(s)
    print(f'{label:<14}{c["steps"]:>7}{nfe(c["steps"], c["corrector_steps"]):>7}{ppl:>10.3f}')

## Notes

- Define arbitrary infilling constraints by editing `make_projection` (e.g. set `input_ids` /
  `input_locs` directly, as shown in the comments of `run_sample_cond.py`).
- Perplexity here covers the **whole** sequence including the pinned prompt; for a cleaner
  signal on the generated span only, restrict the cross-entropy to the non-pinned positions.
- For the absorb variant, corrector unmasks are **irreversible**, so `K=1` is the default;
  the corrector never re-masks or overwrites already-committed (or pinned) positions.
- `corrector_steps=0` reproduces the vanilla conditional sampler bit-for-bit.
- Mean-field caveats (low `t`) apply as in the unconditional case: try `corrector_t_threshold`
  or `update_fraction < 1.0` if infilled spans look locally inconsistent.